In [1]:
import pandas as pd

df = pd.read_csv('../data/raw/diabetic_data.csv')

print(df.shape)
df.head()

(101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [2]:
import numpy as np

df = df.replace('?', np.nan)

df.isnull().sum().sort_values(ascending=False).head(10)

weight               98569
max_glu_serum        96420
A1Cresult            84748
medical_specialty    49949
payer_code           40256
race                  2273
diag_3                1423
diag_2                 358
diag_1                  21
patient_nbr              0
dtype: int64

In [3]:
# Drop columns with unusable amounts of missing data
df = df.drop(columns=['weight', 'payer_code'])

# Fill missing values with an explicit "Missing" category
df['medical_specialty'] = df['medical_specialty'].fillna('Missing')
df['race'] = df['race'].fillna('Missing')

# Confirm the changes
print(df.shape)
df.isnull().sum().sort_values(ascending=False).head(10)

(101766, 48)


max_glu_serum        96420
A1Cresult            84748
diag_3                1423
diag_2                 358
diag_1                  21
encounter_id             0
race                     0
patient_nbr              0
time_in_hospital         0
medical_specialty        0
dtype: int64

In [4]:
df['readmitted'].value_counts()

readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

In [5]:
df['readmitted_binary'] = (df['readmitted'] == '<30').astype(int)

df['readmitted_binary'].value_counts()

readmitted_binary
0    90409
1    11357
Name: count, dtype: int64

In [6]:
imbalance_ratio = df['readmitted_binary'].value_counts(normalize=True)
imbalance_ratio

readmitted_binary
0    0.888401
1    0.111599
Name: proportion, dtype: float64

In [7]:
df['diag_1'].unique()[:20]

<StringArray>
['250.83',    '276',    '648',      '8',    '197',    '414',    '428',
    '398',    '434',  '250.7',    '157',    '518',    '999',    '410',
    '682',    '402',    '737',    '572',    'V57',    '189']
Length: 20, dtype: str

In [8]:
def map_diagnosis(code):
    # Handle missing values first
    if pd.isnull(code):
        return 'Missing'
    
    # Handle V-codes and E-codes (not numeric, so treat separately)
    if code.startswith('V') or code.startswith('E'):
        return 'Other'
    
    # Convert to a number so we can check ranges
    # We only care about the part before the decimal point
    code_num = float(code)
    
    if (390 <= code_num <= 459) or code_num == 785:
        return 'Circulatory'
    elif (460 <= code_num <= 519) or code_num == 786:
        return 'Respiratory'
    elif (520 <= code_num <= 579) or code_num == 787:
        return 'Digestive'
    elif 250 <= code_num < 251:
        return 'Diabetes'
    elif 800 <= code_num <= 999:
        return 'Injury'
    elif 710 <= code_num <= 739:
        return 'Musculoskeletal'
    elif (580 <= code_num <= 629) or code_num == 788:
        return 'Genitourinary'
    elif 140 <= code_num <= 239:
        return 'Neoplasms'
    else:
        return 'Other'

# Apply this function to all three diagnosis columns
df['diag_1_group'] = df['diag_1'].apply(map_diagnosis)
df['diag_2_group'] = df['diag_2'].apply(map_diagnosis)
df['diag_3_group'] = df['diag_3'].apply(map_diagnosis)

df['diag_1_group'].value_counts()

diag_1_group
Circulatory        30437
Other              18172
Respiratory        14423
Digestive           9475
Diabetes            8757
Injury              6974
Genitourinary       5117
Musculoskeletal     4957
Neoplasms           3433
Missing               21
Name: count, dtype: int64

In [9]:
print(df['diag_2_group'].value_counts())
print(df['diag_3_group'].value_counts())

diag_2_group
Circulatory        31881
Other              26553
Diabetes           12794
Respiratory        10895
Genitourinary       8376
Digestive           4170
Neoplasms           2547
Injury              2428
Musculoskeletal     1764
Missing              358
Name: count, dtype: int64
diag_3_group
Circulatory        30306
Other              29195
Diabetes           17157
Respiratory         7358
Genitourinary       6680
Digestive           3930
Injury              1946
Musculoskeletal     1915
Neoplasms           1856
Missing             1423
Name: count, dtype: int64


In [10]:
df.groupby('A1Cresult', dropna=False)['readmitted_binary'].agg(['mean', 'count'])

,mean,count
A1Cresult,,
>7,0.100472,3812
>8,0.098710,8216
Norm,0.096593,4990
NaN,0.114233,84748


In [11]:
df.groupby('age')['readmitted_binary'].agg(['mean', 'count']).sort_index()

,mean,count
age,,
[0-10),0.018634,161
[10-20),0.057887,691
[20-30),0.142426,1657
[30-40),0.112318,3775
[40-50),0.106040,9685
[50-60),0.096662,17256
[60-70),0.111284,22483
[70-80),0.117731,26068
[80-90),0.120835,17197


In [12]:
df.groupby('diag_1_group')['readmitted_binary'].agg(['mean', 'count']).sort_values('mean', ascending=False)

,mean,count
diag_1_group,,
Missing,0.238095,21
Diabetes,0.129839,8757
Injury,0.122455,6974
Other,0.114792,18172
Circulatory,0.114499,30437
Genitourinary,0.108462,5117
Digestive,0.107124,9475
Neoplasms,0.100786,3433
Respiratory,0.097275,14423


In [13]:
numeric_cols = ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 
                 'num_medications', 'number_outpatient', 'number_emergency', 
                 'number_inpatient', 'number_diagnoses']

df[numeric_cols].describe()

,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses
count,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000
mean,4.395987,43.095641,1.339730,16.021844,0.369357,0.197836,0.635566,7.422607
std,2.985108,19.674362,1.705807,8.127566,1.267265,0.930472,1.262863,1.933600
min,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
25%,2.000000,31.000000,0.000000,10.000000,0.000000,0.000000,0.000000,6.000000
50%,4.000000,44.000000,1.000000,15.000000,0.000000,0.000000,0.000000,8.000000
75%,6.000000,57.000000,2.000000,20.000000,0.000000,0.000000,1.000000,9.000000
max,14.000000,132.000000,6.000000,81.000000,42.000000,76.000000,21.000000,16.000000


In [14]:
df.groupby('readmitted_binary')[['number_inpatient', 'number_emergency', 'number_outpatient']].mean()

,number_inpatient,number_emergency,number_outpatient
readmitted_binary,,,
0,0.561648,0.177803,0.360871
1,1.224003,0.357313,0.436911


In [15]:
df.groupby('readmitted_binary')['num_medications'].mean()

readmitted_binary
0    15.911137
1    16.903143
Name: num_medications, dtype: float64

In [16]:
print(df.groupby('change')['readmitted_binary'].agg(['mean', 'count']))
print()
print(df.groupby('diabetesMed')['readmitted_binary'].agg(['mean', 'count']))

            mean  count
change                 
Ch      0.118228  47011
No      0.105908  54755

                 mean  count
diabetesMed                 
No           0.095971  23403
Yes          0.116267  78363


In [17]:
df['discharge_disposition_id'].value_counts().head(15)

discharge_disposition_id
1     60234
3     13954
6     12902
18     3691
2      2128
22     1993
11     1642
5      1184
25      989
4       815
7       623
23      412
13      399
14      372
28      139
Name: count, dtype: int64

In [18]:
death_hospice_codes = [11, 13, 14, 19, 20, 21]

df['discharge_disposition_id'].isin(death_hospice_codes).sum()

np.int64(2423)

In [19]:
print(f"Before: {df.shape}")

df = df[~df['discharge_disposition_id'].isin(death_hospice_codes)]

print(f"After: {df.shape}")

Before: (101766, 52)
After: (99343, 52)


In [20]:
print(f"Total rows: {len(df)}")
print(f"Unique patients: {df['patient_nbr'].nunique()}")
print(f"Rows that are repeat visits: {len(df) - df['patient_nbr'].nunique()}")

Total rows: 99343
Unique patients: 69990
Rows that are repeat visits: 29353


## Note on repeat patient visits

This dataset contains 99,343 encounters from 69,990 unique patients, meaning 29,353 rows (~30%) are repeat visits from patients already represented elsewhere in the data.

The original paper (Strack et al., 2014) addressed this by keeping only each patient's first encounter, reducing their dataset to one row per patient. That approach avoids statistical dependence issues but discards real information, particularly for patients whose *later* visits are the ones that resulted in readmission.

For this project, we instead keep every encounter and address the dependence issue at the modeling stage using **GroupKFold cross-validation**, which ensures all of a single patient's visits stay together in either the training or test split, never both. This prevents data leakage while preserving the full dataset for the model to learn from.

In [21]:
med_columns = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 
               'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 
               'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 
               'miglitol', 'troglitazone', 'tolazamide', 'examide', 
               'citoglipton', 'insulin', 'glyburide-metformin', 
               'glipizide-metformin', 'glimepiride-pioglitazone', 
               'metformin-rosiglitazone', 'metformin-pioglitazone']

for col in med_columns:
    print(df[col].value_counts())
    print()

metformin
No        79499
Steady    18207
Up         1063
Down        574
Name: count, dtype: int64

repaglinide
No        97825
Steady     1368
Up          107
Down         43
Name: count, dtype: int64

nateglinide
No        98654
Steady      654
Up           24
Down         11
Name: count, dtype: int64

chlorpropamide
No        99258
Steady       78
Up            6
Down          1
Name: count, dtype: int64

glimepiride
No        94221
Steady     4609
Up          322
Down        191
Name: count, dtype: int64

acetohexamide
No        99342
Steady        1
Name: count, dtype: int64

glipizide
No        86812
Steady    11219
Up          764
Down        548
Name: count, dtype: int64

glyburide
No        88820
Steady     9162
Up          801
Down        560
Name: count, dtype: int64

tolbutamide
No        99322
Steady       21
Name: count, dtype: int64

pioglitazone
No        92088
Steady     6908
Up          230
Down        117
Name: count, dtype: int64

rosiglitazone
No        93039
Stea

In [22]:
# For patients with more than one visit, check if being readmitted once 
# is associated with being readmitted again

repeat_patients = df[df.duplicated('patient_nbr', keep=False)]

print(f"Encounters from patients with 2+ visits: {len(repeat_patients)}")

# For each patient, calculate their personal readmission rate
patient_readmit_rate = repeat_patients.groupby('patient_nbr')['readmitted_binary'].mean()

print(patient_readmit_rate.describe())

Encounters from patients with 2+ visits: 45694
count    16341.000000
mean         0.176333
std          0.240621
min          0.000000
25%          0.000000
50%          0.000000
75%          0.333333
max          1.000000
Name: readmitted_binary, dtype: float64


In [23]:
cols_to_drop = ['examide', 'citoglipton', 'acetohexamide', 'tolbutamide', 
                 'troglitazone', 'glipizide-metformin', 'glimepiride-pioglitazone',
                 'metformin-rosiglitazone', 'metformin-pioglitazone']

df = df.drop(columns=cols_to_drop)

print(f"Shape after dropping low-variance medication columns: {df.shape}")

Shape after dropping low-variance medication columns: (99343, 43)


In [24]:
med_columns = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 
               'glimepiride', 'glipizide', 'glyburide', 'pioglitazone', 
               'rosiglitazone', 'acarbose', 'miglitol', 'tolazamide', 
               'insulin', 'glyburide-metformin']

# Count how many of these 14 drugs the patient is actually on (i.e. not "No")
df['num_diabetes_meds'] = (df[med_columns] != 'No').sum(axis=1)

# Count how many medications had a dosage change (Up or Down) this encounter
df['num_med_dosage_changes'] = df[med_columns].isin(['Up', 'Down']).sum(axis=1)

df[['num_diabetes_meds', 'num_med_dosage_changes']].describe()

,num_diabetes_meds,num_med_dosage_changes
count,99343.000000,99343.000000
mean,1.186948,0.287368
std,0.922322,0.487861
min,0.000000,0.000000
25%,1.000000,0.000000
50%,1.000000,0.000000
75%,2.000000,1.000000
max,6.000000,4.000000
